In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [456]:
residual_df = pd.read_csv(root/"data"/"clean"/"scorecard"/"clean_residual_FL_programs.csv")
ipedsd_df = pd.read_csv(root/"data"/"clean"/"ipeds"/"clean_ipeds_drivers.csv")

In [457]:
residual_df.head()

,code,unit_id,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,actual,pred,title,4_yr_working_count,error,school_count,confidence,pct_error,score
0,1205,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,22265.0,32497.172,Culinary Arts and Related Services.,28,-10232.171875,6,medium,-0.314863,-0.300975
1,4603,132374,2,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,41177.0,41086.582,Electrical and Power Transmission Installers.,27,90.417969,5,low,0.002201,0.002100
2,4702,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,47325.0,41659.600,"Heating, Air Conditioning, Ventilation and Ref...",48,5665.398438,13,medium,0.135993,0.132757
3,4706,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,42839.0,51169.758,Vehicle Maintenance and Repair Technologies/Te...,28,-8330.757813,24,high,-0.162806,-0.155625
4,5108,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,33081.0,34282.832,Allied Health and Medical Assisting Services.,29,-1201.832031,33,high,-0.035056,-0.033570


In [458]:
major_counts = residual_df["4_yr_working_count"]
school_ids = residual_df["unit_id"]

N = major_counts.groupby(school_ids).transform("sum")
n_eff = (major_counts**2).groupby(school_ids).transform("sum") / N

residual_df["inst_n"]= N
residual_df["inst_effective_n"] = n_eff

In [460]:
w=residual_df["inst_effective_n"]

num=(residual_df["error"]*w).groupby(residual_df["unit_id"]).transform("sum")
dem=w.groupby(residual_df["unit_id"]).transform("sum")

residual_df["error_mean"]=residual_df.groupby("unit_id").transform("median")["error"]
residual_df["inst_wr_mean"] = num/dem

residual_df.head()

C:\Users\sebas\AppData\Local\Temp\ipykernel_14004\1446655189.py:6: FutureWarning:

The default value of numeric_only in DataFrameGroupBy.median is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.



,code,unit_id,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,actual,pred,title,4_yr_working_count,error,school_count,confidence,pct_error,score,inst_n,inst_effective_n,error_mean,inst_wr_mean
0,1205,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,22265.0,32497.172,Culinary Arts and Related Services.,28,-10232.171875,6,medium,-0.314863,-0.300975,206,36.68932,-843.318359,-2415.625
1,4603,132374,2,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,41177.0,41086.582,Electrical and Power Transmission Installers.,27,90.417969,5,low,0.002201,0.002100,206,36.68932,-843.318359,-2415.625
2,4702,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,47325.0,41659.600,"Heating, Air Conditioning, Ventilation and Ref...",48,5665.398438,13,medium,0.135993,0.132757,206,36.68932,-843.318359,-2415.625
3,4706,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,42839.0,51169.758,Vehicle Maintenance and Repair Technologies/Te...,28,-8330.757813,24,high,-0.162806,-0.155625,206,36.68932,-843.318359,-2415.625
4,5108,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1.0,26.0,1,NaN,33081.0,34282.832,Allied Health and Medical Assisting Services.,29,-1201.832031,33,high,-0.035056,-0.033570,206,36.68932,-843.318359,-2415.625


In [461]:
print((residual_df["inst_effective_n"] <= residual_df["inst_n"]).all())
print(residual_df["inst_effective_n"].describe())

True
count    2052.000000
mean      509.169696
std       774.956173
min        21.000000
25%       136.708968
50%       252.802735
75%       483.442177
max      9437.000000
Name: inst_effective_n, dtype: float64


In [462]:
import plotly.express as px

school_df = residual_df[[
    "unit_id",
    "inst_wr_mean",
    "inst_n",
    "inst_effective_n"
]].drop_duplicates()

px.scatter(school_df, x="inst_effective_n",y="inst_wr_mean")

In [463]:
print(school_df["inst_effective_n"].median())
top = school_df.nlargest(10, "inst_wr_mean")
bottom = school_df.nsmallest(10, "inst_wr_mean")

print(top)
print(bottom)

129.0
      unit_id  inst_wr_mean  inst_n  inst_effective_n
1701   428000  50639.886719      92         92.000000
1897   457411  23353.429688     237        237.000000
2000   485272  18140.093750    2655       2655.000000
1777   442295  14702.484375     109        109.000000
1945   481368  10858.177734      24         24.000000
1618   378956  10494.574219     543        543.000000
187    133465  10326.563021    6191       1390.879826
1788   444404   9700.097656      29         29.000000
375    133872   9241.829102     591        136.708968
490    133979   8886.265625      99         24.818182
      unit_id  inst_wr_mean  inst_n  inst_effective_n
1847   449506 -39791.851562     309        309.000000
2001   485342 -16480.093750      22         22.000000
1786   444334 -15099.070312      22         22.000000
2020   490391 -14326.662109      34         34.000000
1478   137953 -12241.578125      26         26.000000
1794   446048  -9918.361328      55         27.727273
1607   369400  -9666.2

In [464]:
k = 250
n = residual_df["inst_effective_n"]
raw = residual_df["inst_wr_mean"]
mu0 = residual_df["inst_wr_mean"].median()

residual_df["adj_inst_wr_mean"] = (
    raw * (n / (n + k)) + mu0 * (k / (n + k))
)

school_df = residual_df[[
    "unit_id",
    "inst_wr_mean",
    "adj_inst_wr_mean",
    "inst_n",
    "inst_effective_n"
]].drop_duplicates()

px.scatter(school_df, x="inst_effective_n",y="adj_inst_wr_mean")


In [465]:
test = residual_df.copy()
median_eff_n = residual_df["inst_effective_n"].median()

def extreme_small_share(df, col):
    top = df.nlargest(20, col)
    bottom = df.nsmallest(20, col)
    
    top_small = (top["inst_effective_n"] < median_eff_n).mean()
    bottom_small = (bottom["inst_effective_n"] < median_eff_n).mean()
    
    return top_small, bottom_small

results = {}

for k in [50, 100, 250, 500]:
    adj = mu0 + (raw - mu0) * (n / (n + k))
    test[f"adj_{k}"] = adj
    
    results[k] = extreme_small_share(test, f"adj_{k}")

results

{50: (0.15, 0.75), 100: (0.15, 0.55), 250: (0.1, 0.1), 500: (0.1, 0.05)}

In [466]:
print(school_df["inst_effective_n"].median())
print(school_df["adj_inst_wr_mean"].median())
print(school_df["adj_inst_wr_mean"].std())


top = school_df.nlargest(10, "adj_inst_wr_mean")
bottom = school_df.nsmallest(10, "adj_inst_wr_mean")

print(top)
print(bottom)

129.0
846.5084427424017
2619.432433656068
      unit_id  inst_wr_mean  adj_inst_wr_mean  inst_n  inst_effective_n
2000   485272  18140.093750      16654.407324    2655       2655.000000
1701   428000  50639.886719      14263.081720      92         92.000000
1897   457411  23353.429688      11814.922394     237        237.000000
187    133465  10326.563021       8886.764476    6191       1390.879826
1618   378956  10494.574219       7462.368437     543        543.000000
1892   457129   6803.156250       6650.200258    9437       9437.000000
61     132709   6144.857244       5127.198742    1923       1044.255330
1777   442295  14702.484375       5074.304086     109        109.000000
210    133508   5166.555664       4199.603409    1761        859.190801
1649   406024   4517.148437       3969.742199    2925       1412.718974
      unit_id  inst_wr_mean  adj_inst_wr_mean  inst_n  inst_effective_n
1847   449506 -39791.851562     -21603.895819     309        309.000000
1607   369400  -9666.2

In [467]:
save(residual_df,clean=1,file_name="inst_residual_FL")


In [239]:
ipedsd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6909 entries, 0 to 6908
Data columns (total 31 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   6909 non-null   int64  
 1   year                                      6909 non-null   int64  
 2   code                                      6909 non-null   int64  
 3   credential_level                          6909 non-null   int64  
 4   program_completers                        6909 non-null   int64  
 5   program_completers_log                    6909 non-null   float64
 6   program_completer_share_within_school     6906 non-null   float64
 7   instruction_salary_pct                    0 non-null      float64
 8   academic_support_salary_pct               0 non-null      float64
 9   student_services_salary_pct               0 non-null      float64
 10  research_salary_pct                 

In [240]:
ipedsd_df=ipedsd_df.drop(columns=['instruction_salary_pct', 'academic_support_salary_pct',
       'student_services_salary_pct', 'research_salary_pct'])

In [241]:
ipedsd_df.head()

,unit_id,year,code,credential_level,program_completers,program_completers_log,program_completer_share_within_school,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share
0,132374,2020,1101,1,14,2.708050,0.016588,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
1,132374,2020,1102,1,10,2.397895,0.011848,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
2,132374,2020,1108,1,40,3.713572,0.047393,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
3,132374,2020,1109,1,34,3.555348,0.040284,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
4,132374,2020,1205,1,47,3.871201,0.055687,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN


In [544]:
targ="inst_wr_mean"
merge_df=ipedsd_df.merge(residual_df[["unit_id","code","credential_level",targ,"school_name"]], on = ["unit_id","code","credential_level"], how="right")

In [545]:
model_df=merge_df.copy()
print("ipedsd_df shape:", ipedsd_df.shape)
print("residual_df shape:", residual_df.shape)
print("model_df shape:", model_df.shape)

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

ipedsd_df shape: (6909, 31)
residual_df shape: (2052, 29)
model_df shape: (2052, 33)
model_df columns:
['academic_support_salary_pct', 'career_counseling', 'code', 'credential_level', 'employment_services', 'endowment_per_fte', 'equity_ratio', 'inst_wr_mean', 'instruction_expense_pct', 'instruction_salary_pct', 'instructional_fte_per_student', 'instructional_share_of_staff', 'instructional_staff_long_contract', 'instructional_staff_long_contract_share', 'instructional_staff_short_contract', 'instructional_staff_short_contract_share', 'instructional_staff_total', 'irps_fte_per_student', 'no_ap_credit', 'placement_services', 'program_completer_share_within_school', 'program_completers', 'program_completers_log', 'research_expense_pct', 'research_salary_pct', 'research_share_of_staff', 'school_name', 'staff_per_student', 'student_service_expense_pct', 'student_services_salary_pct', 'study_abroad', 'unit_id', 'year']


In [546]:
model_df=model_df.drop(columns=[
    'year','program_completer_share_within_school','program_completers',
    'program_completers_log',"credential_level",'code',
    # 'unit_id',
    'school_name'
    ]
)

In [547]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2052 entries, 0 to 2051
Data columns (total 26 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   2052 non-null   int64  
 1   instruction_salary_pct                    0 non-null      float64
 2   academic_support_salary_pct               0 non-null      float64
 3   student_services_salary_pct               0 non-null      float64
 4   research_salary_pct                       0 non-null      float64
 5   no_ap_credit                              1922 non-null   float64
 6   study_abroad                              1922 non-null   float64
 7   career_counseling                         1922 non-null   float64
 8   employment_services                       1922 non-null   float64
 9   placement_services                        1922 non-null   float64
 10  instruction_expense_pct             

In [548]:
display("Repeated columns within instituions",(model_df.groupby("unit_id").nunique() > 1).sum())
display("Feature correlation to target variable",model_df.corr()[targ].sort_values())

'Repeated columns within instituions'

instruction_salary_pct                      0
academic_support_salary_pct                 0
student_services_salary_pct                 0
research_salary_pct                         0
no_ap_credit                                0
study_abroad                                0
career_counseling                           0
employment_services                         0
placement_services                          0
instruction_expense_pct                     0
research_expense_pct                        0
student_service_expense_pct                 0
endowment_per_fte                           0
equity_ratio                                0
staff_per_student                           0
instructional_fte_per_student               0
irps_fte_per_student                        0
instructional_share_of_staff                0
research_share_of_staff                     0
instructional_staff_total                   0
instructional_staff_short_contract          0
instructional_staff_long_contract 

'Feature correlation to target variable'

unit_id                                    -0.159859
equity_ratio                               -0.131486
instructional_share_of_staff               -0.118153
student_service_expense_pct                -0.095325
no_ap_credit                               -0.081597
instruction_expense_pct                    -0.067790
instructional_staff_short_contract_share   -0.014400
placement_services                          0.001235
instructional_staff_short_contract          0.003389
instructional_staff_long_contract_share     0.014400
career_counseling                           0.022871
study_abroad                                0.053603
employment_services                         0.057682
research_share_of_staff                     0.079113
instructional_fte_per_student               0.100188
staff_per_student                           0.140405
irps_fte_per_student                        0.146493
instructional_staff_long_contract           0.161214
instructional_staff_total                   0.

In [549]:
inst_model_df = model_df.drop_duplicates(subset="unit_id").reset_index(drop=True)
inst_model_df=inst_model_df.drop(columns="unit_id")
print("inst_model_df shape:", inst_model_df.shape)
display(inst_model_df.head())

inst_model_df shape: (237, 25)


,instruction_salary_pct,academic_support_salary_pct,student_services_salary_pct,research_salary_pct,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share,inst_wr_mean
0,NaN,NaN,NaN,NaN,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN,-2415.625000
1,NaN,NaN,NaN,NaN,0.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,0.163673,0.065868,0.065868,0.402439,0.0,46.0,0.0,46.0,1.0,0.0,-7158.445313
2,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.102841,0.043977,0.043977,0.427624,0.0,550.0,0.0,550.0,1.0,0.0,4972.134001
3,NaN,NaN,NaN,NaN,0.0,1.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,0.131015,0.054331,0.054331,0.414692,0.0,304.0,0.0,304.0,1.0,0.0,-8228.132254
4,NaN,NaN,NaN,NaN,0.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0.153951,0.044800,0.044800,0.291005,0.0,244.0,0.0,244.0,1.0,0.0,-1361.238281


In [550]:
display(inst_model_df.describe())
display("Feature correlation to target variable",inst_model_df.corr()[targ].sort_values())
display(inst_model_df.info())

# drop_cols=[
#     "instruction_expense_pct",
#     "student_service_expense_pct",
#     "career_counseling",
#     "employment_services",
# ]

,instruction_salary_pct,academic_support_salary_pct,student_services_salary_pct,research_salary_pct,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share,inst_wr_mean
count,0.0,0.0,0.0,0.0,215.000000,215.000000,215.000000,215.000000,215.000000,68.000000,68.000000,68.000000,39.000000,39.000000,214.000000,214.000000,214.000000,215.000000,215.000000,215.000000,215.000000,215.000000,112.000000,112.000000,237.000000
mean,NaN,NaN,NaN,NaN,0.451163,0.246512,0.916279,0.730233,0.906977,43.470588,2.088235,9.367647,6948.282051,65.102564,0.100038,0.040879,0.041561,0.441886,0.001667,237.069767,1.423256,235.646512,0.990210,0.009790,711.908489
std,NaN,NaN,NaN,NaN,0.498770,0.431986,0.277615,0.444875,0.291143,12.449864,5.525343,4.424843,10204.337166,14.615772,0.095635,0.033058,0.033831,0.124798,0.008103,614.303397,16.365065,612.671205,0.043422,0.043422,6182.632521
min,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,15.000000,0.000000,0.000000,227.000000,37.000000,0.016738,0.007823,0.007823,0.131088,0.000000,0.000000,0.000000,0.000000,0.708333,0.000000,-39791.851562
25%,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,0.000000,1.000000,36.000000,0.000000,6.000000,1677.500000,58.000000,0.052814,0.022413,0.022659,0.359526,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,-1586.958984
50%,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,1.000000,1.000000,39.000000,0.000000,10.000000,4223.000000,66.000000,0.076798,0.034701,0.034701,0.439024,0.000000,2.000000,0.000000,2.000000,1.000000,0.000000,770.225000
75%,NaN,NaN,NaN,NaN,1.000000,0.000000,1.000000,1.000000,1.000000,52.000000,0.000000,13.000000,7347.000000,75.500000,0.112066,0.052169,0.052488,0.520274,0.000000,177.000000,0.000000,177.000000,1.000000,0.000000,2762.884440
max,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,73.000000,25.000000,18.000000,55596.000000,89.000000,0.937500,0.375000,0.375000,0.833333,0.063864,5044.000000,238.000000,5044.000000,1.000000,0.291667,50639.886719


'Feature correlation to target variable'

endowment_per_fte                          -0.251383
equity_ratio                               -0.044504
no_ap_credit                               -0.030244
instructional_share_of_staff               -0.015048
instructional_staff_long_contract_share    -0.009343
instructional_staff_short_contract          0.007640
instructional_staff_short_contract_share    0.009343
career_counseling                           0.019684
research_share_of_staff                     0.029737
study_abroad                                0.036486
instructional_staff_long_contract           0.048666
instructional_staff_total                   0.048741
instruction_expense_pct                     0.049968
employment_services                         0.081432
placement_services                          0.102628
research_expense_pct                        0.105546
staff_per_student                           0.117727
instructional_fte_per_student               0.123169
student_service_expense_pct                 0.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   instruction_salary_pct                    0 non-null      float64
 1   academic_support_salary_pct               0 non-null      float64
 2   student_services_salary_pct               0 non-null      float64
 3   research_salary_pct                       0 non-null      float64
 4   no_ap_credit                              215 non-null    float64
 5   study_abroad                              215 non-null    float64
 6   career_counseling                         215 non-null    float64
 7   employment_services                       215 non-null    float64
 8   placement_services                        215 non-null    float64
 9   instruction_expense_pct                   68 non-null     float64
 10  research_expense_pct                  

None

In [551]:
display(inst_model_df.describe())
display("Feature correlation to target variable",inst_model_df.corr()[targ].sort_values())
display(inst_model_df.info())

,instruction_salary_pct,academic_support_salary_pct,student_services_salary_pct,research_salary_pct,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share,inst_wr_mean
count,0.0,0.0,0.0,0.0,215.000000,215.000000,215.000000,215.000000,215.000000,68.000000,68.000000,68.000000,39.000000,39.000000,214.000000,214.000000,214.000000,215.000000,215.000000,215.000000,215.000000,215.000000,112.000000,112.000000,237.000000
mean,NaN,NaN,NaN,NaN,0.451163,0.246512,0.916279,0.730233,0.906977,43.470588,2.088235,9.367647,6948.282051,65.102564,0.100038,0.040879,0.041561,0.441886,0.001667,237.069767,1.423256,235.646512,0.990210,0.009790,711.908489
std,NaN,NaN,NaN,NaN,0.498770,0.431986,0.277615,0.444875,0.291143,12.449864,5.525343,4.424843,10204.337166,14.615772,0.095635,0.033058,0.033831,0.124798,0.008103,614.303397,16.365065,612.671205,0.043422,0.043422,6182.632521
min,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,15.000000,0.000000,0.000000,227.000000,37.000000,0.016738,0.007823,0.007823,0.131088,0.000000,0.000000,0.000000,0.000000,0.708333,0.000000,-39791.851562
25%,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,0.000000,1.000000,36.000000,0.000000,6.000000,1677.500000,58.000000,0.052814,0.022413,0.022659,0.359526,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,-1586.958984
50%,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,1.000000,1.000000,39.000000,0.000000,10.000000,4223.000000,66.000000,0.076798,0.034701,0.034701,0.439024,0.000000,2.000000,0.000000,2.000000,1.000000,0.000000,770.225000
75%,NaN,NaN,NaN,NaN,1.000000,0.000000,1.000000,1.000000,1.000000,52.000000,0.000000,13.000000,7347.000000,75.500000,0.112066,0.052169,0.052488,0.520274,0.000000,177.000000,0.000000,177.000000,1.000000,0.000000,2762.884440
max,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,73.000000,25.000000,18.000000,55596.000000,89.000000,0.937500,0.375000,0.375000,0.833333,0.063864,5044.000000,238.000000,5044.000000,1.000000,0.291667,50639.886719


'Feature correlation to target variable'

endowment_per_fte                          -0.251383
equity_ratio                               -0.044504
no_ap_credit                               -0.030244
instructional_share_of_staff               -0.015048
instructional_staff_long_contract_share    -0.009343
instructional_staff_short_contract          0.007640
instructional_staff_short_contract_share    0.009343
career_counseling                           0.019684
research_share_of_staff                     0.029737
study_abroad                                0.036486
instructional_staff_long_contract           0.048666
instructional_staff_total                   0.048741
instruction_expense_pct                     0.049968
employment_services                         0.081432
placement_services                          0.102628
research_expense_pct                        0.105546
staff_per_student                           0.117727
instructional_fte_per_student               0.123169
student_service_expense_pct                 0.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   instruction_salary_pct                    0 non-null      float64
 1   academic_support_salary_pct               0 non-null      float64
 2   student_services_salary_pct               0 non-null      float64
 3   research_salary_pct                       0 non-null      float64
 4   no_ap_credit                              215 non-null    float64
 5   study_abroad                              215 non-null    float64
 6   career_counseling                         215 non-null    float64
 7   employment_services                       215 non-null    float64
 8   placement_services                        215 non-null    float64
 9   instruction_expense_pct                   68 non-null     float64
 10  research_expense_pct                  

None

In [552]:
import plotly.express as px
px.histogram(inst_model_df[targ])

In [553]:
inst_model_df[targ].skew()

1.1332330717939543

In [554]:
inst_model_df[targ].describe()

count      237.000000
mean       711.908489
std       6182.632521
min     -39791.851562
25%      -1586.958984
50%        770.225000
75%       2762.884440
max      50639.886719
Name: inst_wr_mean, dtype: float64

In [555]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [556]:
# -------------------
# 1. Define target
# -------------------
raw_target_col = targ

# Drop target and target-like columns from X
drop_cols = [
    raw_target_col,
    "inst_wr_mean",   # important: likely leakage / near-leakage
    "unit_id"         # ID, not a feature
]

X = inst_model_df.drop(columns=drop_cols, errors="ignore").copy()
y = pd.to_numeric(inst_model_df[raw_target_col], errors="coerce").copy()

# Force predictors numeric
X = X.apply(pd.to_numeric, errors="coerce")

# -------------------
# 2. Train/test split
# -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# -------------------
# 3. Clip target using TRAINING quantiles only
# -------------------
lower = y_train.quantile(0.01)
upper = y_train.quantile(0.99)

y_train_clip = y_train.clip(lower, upper)
y_test_clip = y_test.clip(lower, upper)

# -------------------
# 4. Recompute numeric columns
# -------------------
num_cols = X_train.columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols)
])

# -------------------
# 5. Ridge model
# -------------------
ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Ridge())
])

ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipe,
    transformer=PowerTransformer(method="yeo-johnson", standardize=False)
)

ridge_param_grid = {
    "regressor__reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0, 100.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_grid = GridSearchCV(
    estimator=ridge_model,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ridge_grid.fit(X_train, y_train_clip)

ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(X_test)

print("RIDGE")
print("Best params:", ridge_grid.best_params_)
print("Best CV MAE:", round(-ridge_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, ridge_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, ridge_preds), 4))

print("Test MAE (vs clipped y_test):", round(mean_absolute_error(y_test_clip, ridge_preds), 4))
print("Test R2 (vs clipped y_test):", round(r2_score(y_test_clip, ridge_preds), 4))

# feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
# print(feature_names)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
RIDGE
Best params: {'regressor__reg__alpha': 100.0}
Best CV MAE: 3549.5845
Test MAE (vs unclipped y_test): 3714.058
Test R2 (vs unclipped y_test): -0.0361
Test MAE (vs clipped y_test): 2973.9562
Test R2 (vs clipped y_test): -0.063


In [557]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=10000))
])

enet_model = TransformedTargetRegressor(
    regressor=enet_pipe,
    transformer=PowerTransformer(method="yeo-johnson", standardize=False)
)

enet_param_grid = {
    "regressor__reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "regressor__reg__l1_ratio": [0.005,0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    estimator=enet_model,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(X_train, y_train_clip)

enet_best = enet_grid.best_estimator_
enet_preds = enet_best.predict(X_test)

print("\nELASTIC NET")
print("Best params:", enet_grid.best_params_)
print("Best CV MAE:", round(-enet_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, enet_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, enet_preds), 4))
print("Test MAE (vs clipped y_test):", round(mean_absolute_error(y_test_clip, enet_preds), 4))
print("Test R2 (vs clipped y_test):", round(r2_score(y_test_clip, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits

ELASTIC NET
Best params: {'regressor__reg__alpha': 50.0, 'regressor__reg__l1_ratio': 0.005}
Best CV MAE: 3356.8418
Test MAE (vs unclipped y_test): 3477.8069
Test R2 (vs unclipped y_test): -0.0201
Test MAE (vs clipped y_test): 2737.705
Test R2 (vs clipped y_test): -0.0096


In [521]:
N_RUNS = 20

coef_list = []

for i in range(N_RUNS):

    # -------------------
    # Split
    # -------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE + i
    )

    # -------------------
    # Rebuild preprocessor (IMPORTANT)
    # -------------------
    num_cols = X_train.columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols)
    ])

    # -------------------
    # Model
    # -------------------
    enet_pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", ElasticNet(alpha=10.0, l1_ratio=0.7, max_iter=10000))
    ])

    model = TransformedTargetRegressor(
        regressor=enet_pipe,
        transformer=PowerTransformer(method="yeo-johnson", standardize=False)
    )

    # -------------------
    # Fit
    # -------------------
    model.fit(X_train, y_train)

    # -------------------
    # Extract coefficients
    # -------------------
    reg_pipe = model.regressor_
    feature_names = reg_pipe.named_steps["preprocessor"].get_feature_names_out()

    coefs = reg_pipe.named_steps["reg"].coef_

    coef_series = pd.Series(coefs, index=feature_names)
    coef_list.append(coef_series)

# -------------------
# Combine results
# -------------------
coef_df = pd.concat(coef_list, axis=1)
coef_df.columns = [f"run_{i}" for i in range(N_RUNS)]

# -------------------
# Summary stats
# -------------------
summary = pd.DataFrame({
    "mean_coef": coef_df.mean(axis=1),
    "std_coef": coef_df.std(axis=1),
    "nonzero_pct": (coef_df != 0).mean(axis=1)
}).sort_values("mean_coef")

print(summary)

                                                mean_coef   std_coef  \
num__endowment_per_fte                         -87.407409  56.322449   
num__instructional_staff_long_contract_share    -7.858571  15.727168   
num__instructional_share_of_staff               -6.544582  39.284984   
num__no_ap_credit                               -6.507262  55.093769   
num__equity_ratio                               -5.934732  24.890554   
num__instructional_staff_short_contract          1.252435   5.299820   
num__instructional_staff_short_contract_share    7.858591  15.727169   
num__instruction_expense_pct                    15.949076  26.865473   
num__career_counseling                          16.396551  43.216617   
num__research_share_of_staff                    17.966182   9.665658   
num__research_expense_pct                       19.171351   8.407919   
num__study_abroad                               27.143051  31.224693   
num__instructional_staff_total                  42.310282  12.12

In [406]:
y = inst_model_df["adj_inst_wr_mean"]
y_pred = [y.mean()] * len(y)

mean_absolute_error(y, y_pred)

1230.9992516305804

In [522]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

model.fit(X_train, y_train)
preds = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

MAE: 5844.822276145584
R2: -0.09730225849508001


In [529]:
from sklearn.linear_model import LogisticRegression
# -------------------
# 1. Target (top vs bottom)
# -------------------
y_raw = inst_model_df["inst_wr_mean"].copy()

low = y_raw.quantile(0.25)
high = y_raw.quantile(0.75)

mask = (y_raw <= low) | (y_raw >= high)

# -------------------
# 2. Use ALL columns except target
# -------------------
X_clf = inst_model_df.drop(columns=["inst_wr_mean"], errors="ignore").loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)

# force everything numeric
X_clf = X_clf.apply(pd.to_numeric, errors="coerce")

# -------------------
# 3. Train/test split
# -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# -------------------
# 4. Preprocessing (CRITICAL)
# -------------------
num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

# -------------------
# 5. Model
# -------------------
clf = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", LogisticRegression(max_iter=10000))
])

# -------------------
# 6. Fit
# -------------------
clf.fit(X_train, y_train)

# -------------------
# 7. Evaluate
# -------------------
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("ROC AUC:", round(roc_auc_score(y_test, probs), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

Accuracy: 0.4583
ROC AUC: 0.5208

Confusion Matrix:
 [[4 8]
 [5 7]]

Classification Report:
               precision    recall  f1-score   support

           0       0.44      0.33      0.38        12
           1       0.47      0.58      0.52        12

    accuracy                           0.46        24
   macro avg       0.46      0.46      0.45        24
weighted avg       0.46      0.46      0.45        24



In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

clf_hgb = HistGradientBoostingClassifier(
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

clf_hgb.fit(X_train, y_train)

preds_hgb = clf_hgb.predict(X_test)
probs_hgb = clf_hgb.predict_proba(X_test)[:, 1]

print("HGB Accuracy:", round(accuracy_score(y_test, preds_hgb), 4))
print("HGB ROC AUC:", round(roc_auc_score(y_test, probs_hgb), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds_hgb))
print("\nClassification Report:\n", classification_report(y_test, preds_hgb))

HGB Accuracy: 0.4167
HGB ROC AUC: 0.375

Confusion Matrix:
 [[4 8]
 [6 6]]

Classification Report:
               precision    recall  f1-score   support

           0       0.40      0.33      0.36        12
           1       0.43      0.50      0.46        12

    accuracy                           0.42        24
   macro avg       0.41      0.42      0.41        24
weighted avg       0.41      0.42      0.41        24

